In [0]:
"""
================================================================================
 load_to_sqlserver.py   â€”  Bulk loader for the clinical-trial SOURCE tables
================================================================================
Loads the generated SDTM CSV files into the SQL Server [src].* tables, honouring
FULL vs INCREMENTAL load semantics. Use the SAME script for every run:

    Run 1 (initial):   point DATA_DIR at data/raw     -> full history loaded
    Run 2 (delta):     point DATA_DIR at data/run2     -> upsert new/changed rows

How each mode behaves
---------------------
  FULL          TRUNCATE the table, then load every row from the file.
                (Use for DS, TA, TS â€” the run-2 file must contain the FULL set.)
  INCREMENTAL   MERGE (upsert) by primary key:
                  * new keys     -> INSERT
                  * matched keys -> UPDATE
                  * MODIFIED_TS is stamped = SYSUTCDATETIME() on insert/update,
                    so ADF's watermark picks the row up on the next run.

All CSV values are staged as text (NVARCHAR) and SQL Server converts them to the
real column types during the set-based INSERT/MERGE â€” this avoids pyodbc numeric
binding errors.

Requirements:  pip install pandas pyodbc   + ODBC Driver 17/18 for SQL Server.
Connection  :  Windows Authentication (Trusted_Connection=yes).
================================================================================
"""

import os
import sys
import math
import pandas as pd
import pyodbc

# -----------------------------------------------------------------------------
# 1. CONNECTION  (Windows Authentication)  â€” EDIT THESE
# -----------------------------------------------------------------------------
SERVER   = r"localhost\SQLEXPRESS"   # named instance (from your Connection Properties)
DATABASE = "dev_clintrail_edc_src"           # your database
DRIVER   = "{ODBC Driver 17 for SQL Server}"   # use 18 if that's what you have
SCHEMA   = "src"
WATERMARK_COL = "MODIFIED_TS"

# Folder holding the CSVs for THIS run (run 1 = data/raw, run 2 = data/run2)
BASE_DIR = "/Workspace/Users/barshilekiran14@gmail.com/IntelliBi_Databricks/IntellibiDatabrickspro/Data_factory_data"
DATA_DIR = os.path.join(BASE_DIR, "data", "raw")     # <-- change to data/run2 for run 2

# -----------------------------------------------------------------------------
# 2. LOAD CONFIG  â€” one entry per table: (mode, primary-key columns)
# -----------------------------------------------------------------------------
CONFIG = {
    "DM": ("incremental", ["USUBJID"]),
    "SV": ("incremental", ["USUBJID", "VISITNUM"]),
    "VS": ("incremental", ["USUBJID", "VSSEQ"]),
    "LB": ("incremental", ["USUBJID", "LBSEQ"]),
    "EG": ("incremental", ["USUBJID", "EGSEQ"]),
    "EC": ("incremental", ["USUBJID", "ECSEQ"]),
    "CM": ("incremental", ["USUBJID", "CMSEQ"]),
    "DS": ("full",        ["USUBJID", "DSSEQ"]),
    "TA": ("full",        ["STUDYID", "ARMCD", "TAETORD"]),
    "TS": ("full",        ["STUDYID", "TSPARMCD"]),
}


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def connect():
    cs = (f"DRIVER={DRIVER};SERVER={SERVER};DATABASE={DATABASE};"
          f"Trusted_Connection=yes;")
    return pyodbc.connect(cs, autocommit=False)


def table_columns(cur, table):
    cur.execute(
        "SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS "
        "WHERE TABLE_SCHEMA=? AND TABLE_NAME=? ORDER BY ORDINAL_POSITION",
        SCHEMA, table)
    return [r[0] for r in cur.fetchall()]


def clean_rows(df, cols):
    """DataFrame -> list of tuples of (str | None). Guarantees no NaN/float
    ever reaches pyodbc (that is what triggered the type-inference errors)."""
    rows = []
    for rec in df[cols].itertuples(index=False, name=None):
        out = []
        for v in rec:
            if v is None:
                out.append(None)
            elif isinstance(v, float) and math.isnan(v):
                out.append(None)
            elif isinstance(v, str) and v.strip() == "":
                out.append(None)
            else:
                out.append(str(v))
        rows.append(tuple(out))
    return rows


def make_staging(cur, stg, cols):
    """All-text temp table so string CSV values bind cleanly; SQL Server
    converts text -> real column types during the INSERT / MERGE."""
    cur.execute(f"IF OBJECT_ID('tempdb..{stg}') IS NOT NULL DROP TABLE {stg}")
    coldefs = ",".join(f"[{c}] NVARCHAR(4000) NULL" for c in cols)
    cur.execute(f"CREATE TABLE {stg} ({coldefs})")


def fill_staging(cur, stg, cols, df):
    if df.empty:
        return
    marks = ",".join(["?"] * len(cols))
    sql = f"INSERT INTO {stg} ([{'],['.join(cols)}]) VALUES ({marks})"
    # size each text parameter to its real max width -> fast_executemany is both
    # fast and cannot misinfer a column as float/numeric.
    sizes = []
    for c in cols:
        try:
            w = int(df[c].astype(str).str.len().max())
        except Exception:
            w = 1
        sizes.append((pyodbc.SQL_WVARCHAR, max(1, min(w, 4000)), 0))
    cur.fast_executemany = True
    cur.setinputsizes(sizes)
    data = clean_rows(df, cols)
    for i in range(0, len(data), 20000):       # chunk to keep memory bounded
        cur.executemany(sql, data[i:i + 20000])
    cur.setinputsizes(None)


def load_full(cur, table, keys, df, tcols):
    cols = [c for c in df.columns if c in tcols and c != WATERMARK_COL]
    stg = f"#stg_{table}"
    make_staging(cur, stg, cols)
    fill_staging(cur, stg, cols, df)

    cur.execute(f"TRUNCATE TABLE {SCHEMA}.{table}")
    ins_cols = list(cols) + ([WATERMARK_COL] if WATERMARK_COL in tcols else [])
    sel = ",".join(f"[{c}]" for c in cols) + \
          (",SYSUTCDATETIME()" if WATERMARK_COL in tcols else "")
    cur.execute(f"INSERT INTO {SCHEMA}.{table} ([{'],['.join(ins_cols)}]) "
                f"SELECT {sel} FROM {stg}")
    n = cur.rowcount
    cur.execute(f"DROP TABLE {stg}")
    return f"FULL   loaded {n} rows (truncate + insert)"


def load_incremental(cur, table, keys, df, tcols):
    cols = [c for c in df.columns if c in tcols and c != WATERMARK_COL]
    non_key = [c for c in cols if c not in keys]
    stg = f"#stg_{table}"
    make_staging(cur, stg, cols)
    fill_staging(cur, stg, cols, df)

    ins_cols = list(cols) + ([WATERMARK_COL] if WATERMARK_COL in tcols else [])
    ins_sel  = ",".join(f"[{c}]" for c in cols) + \
               (",SYSUTCDATETIME()" if WATERMARK_COL in tcols else "")

    # FAST PATH: empty target (run 1) -> single set-based INSERT, no MERGE
    cur.execute(f"SELECT TOP 1 1 FROM {SCHEMA}.{table}")
    if cur.fetchone() is None:
        cur.execute(f"INSERT INTO {SCHEMA}.{table} ([{'],['.join(ins_cols)}]) "
                    f"SELECT {ins_sel} FROM {stg}")
        n = cur.rowcount
        cur.execute(f"DROP TABLE {stg}")
        return f"INCR   inserted {n} rows (initial fast load)"

    on_clause  = " AND ".join(f"t.[{k}]=s.[{k}]" for k in keys)
    set_clause = ", ".join(f"t.[{c}]=s.[{c}]" for c in non_key)
    if WATERMARK_COL in tcols:
        set_clause += f", t.[{WATERMARK_COL}]=SYSUTCDATETIME()"
    ins_vals = ",".join(f"s.[{c}]" for c in cols) + \
               (",SYSUTCDATETIME()" if WATERMARK_COL in tcols else "")

    # delta files hold only changed + new rows: update every match, insert rest.
    merge = f"""
    MERGE {SCHEMA}.{table} AS t
    USING {stg} AS s ON {on_clause}
    WHEN MATCHED
        THEN UPDATE SET {set_clause}
    WHEN NOT MATCHED BY TARGET
        THEN INSERT ([{'],['.join(ins_cols)}]) VALUES ({ins_vals});
    """
    cur.execute(merge)
    affected = cur.rowcount
    cur.execute(f"DROP TABLE {stg}")
    return f"INCR   upserted ~{affected} rows (insert new + update matched)"


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def main():
    print(f"Loading from: {DATA_DIR}")
    print(f"Target      : {SERVER} / {DATABASE} / [{SCHEMA}]\n")
    conn = connect()
    cur = conn.cursor()
    total = 0
    try:
        for table, (mode, keys) in CONFIG.items():
            path = os.path.join(DATA_DIR, f"{table}.csv")
            if not os.path.exists(path):
                print(f"  {table:4s} -- no file ({table}.csv) â€” skipped")
                continue
            df = pd.read_csv(path, dtype=str)
            tcols = table_columns(cur, table)
            if not tcols:
                print(f"  {table:4s} -- table src.{table} not found â€” run 04_source_tables.sql first")
                continue
            if mode == "full":
                msg = load_full(cur, table, keys, df, tcols)
            else:
                msg = load_incremental(cur, table, keys, df, tcols)
            conn.commit()
            total += len(df)
            print(f"  {table:4s} -- {msg}")
        print(f"\nDone. {total} source rows processed. Committed.")
    except Exception as e:
        conn.rollback()
        print(f"\nERROR (rolled back): {e}")
        raise
    finally:
        cur.close()
        conn.close()


if __name__ == "__main__":
    # optional: python load_to_sqlserver.py ..\data\run2
    if len(sys.argv) > 1:
        DATA_DIR = os.path.abspath(sys.argv[1])
    main()